# Contextual axis vs. communication subspace: does subtracting the task-condition mean matter?

A variant of `contextual_vs_communication_one_session_3regressors.ipynb` with **no
nuisance regression**, restricted to the **quiescent period** before stimulus onset. It
asks one question: when the communication subspace is estimated from *residual*
activity, how does its alignment with the contextual (task) axis change depending on
whether the task-condition mean is subtracted as well?

Pipeline, per session:

1. **Qualified cells**: same selection as the 3-regressor notebook -- QC-pass single
   units (`is_qc_pass` and `decoder_label == 'sua'`), areas with at least `MIN_UNITS`
   of them, every area subsampled to exactly `MIN_UNITS` with the same seed. The unit
   sets are therefore identical to that notebook's.
2. **Base activity**: per-trial, per-unit firing rate over the quiescent window
   (`quiescent_start_time` -> `quiescent_stop_time`), z-scored per unit over all
   stimulus trials. Nothing is regressed out.
3. **Residual variants** (`RESIDUAL_VARIANTS`): the base activity minus a per-unit
   condition mean, where the condition is defined by zero or more trial-table columns:
   - `grand_mean` -- no condition columns: only the per-unit mean over all trials is
     removed (which z-scoring already did). Context is fully retained.
   - `context_mean` -- the mean per rewarded modality (visual / auditory block) is
     removed. The across-context difference is gone; what remains is trial-to-trial
     variability within a context.
4. **Contextual axis**: one LDA axis per area, fit on the `grand_mean` activity. It is
   shared across variants -- it has to be, since subtracting the context mean makes the
   two class means identical and LDA degenerate.
5. **Communication subspace**: for every ordered area pair and every variant, ridge
   regression from source to target on that variant's residual, SVD of the prediction,
   top singular vectors to 90% of predictable variance, orthonormalized in the source
   area's unit space. Residuals are **not** re-standardized before the fit: the
   contextual axis is expressed in the base (z-scored) unit coordinates, and
   re-scaling each unit's residual would put the subspace in a different coordinate
   system and silently distort the cosine similarity.
6. **Comparison**: cosine similarity (norm of the contextual axis projected onto the
   subspace) for each variant, side by side and as paired differences, with the same
   three permutation nulls as the 3-regressor notebook (label shuffle, trial shuffle,
   random axis), plus a direct measure of how much the subspace itself moves between
   variants.

Reads `DATACUBE_ROOT` and `SESSION_ID` from the environment like the other notebooks,
so it runs unchanged locally and via `run_capsule.py`.

# Data loading

In [ ]:
import os
from pathlib import Path

import numpy as np
import numpy.random as npr
import pandas as pd
import pynwb
from tqdm import tqdm
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
CAPSULE_ROOT = Path("/root/capsule/data/dynamicrouting_datacube")
LOCAL_ROOT = Path.cwd().parents[1] / "data" / "dynamicrouting_datacube"  # local bundle layout
root = Path(os.environ.get("DATACUBE_ROOT", CAPSULE_ROOT if CAPSULE_ROOT.exists() else LOCAL_ROOT))
print(f'DATACUBE_ROOT = {root}')
session_id = os.environ.get("SESSION_ID", "743199_2024-12-05")
print(f'session_id = {session_id}')

nwb_path = None
for d in root.iterdir():
    candidate = d / f"{session_id}.nwb.zarr"
    if candidate.exists():
        nwb_path = candidate
        print(nwb_path)
        break
assert nwb_path is not None, f"no {session_id}.nwb.zarr under {root}"

In [ ]:
# should take about a minute
session = pynwb.read_nwb(nwb_path)

trials = session.trials[:]
units = session.units[:]
print(f"{len(trials)} trials, {len(units)} units")

# Quiescent-period firing rate per trial

Stimulus trials only (catch trials excluded). No running-speed filter is applied, since
nothing is regressed out here -- so the trial count can differ by a few trials from
the 3-regressor notebook, which drops trials with no running sample in the window.

In [ ]:
reg_trials = trials[trials['stim_name'] != 'catch'].reset_index(drop=True)
window_starts = reg_trials['quiescent_start_time'].values
window_stops = reg_trials['quiescent_stop_time'].values
print(f"{len(reg_trials)} / {len(trials)} stimulus trials; quiescent window "
      f"{np.median(window_stops - window_starts):.2f} s (median)")


def trial_interval_firing_rate(spike_times, starts, stops):
    spike_times = np.asarray(spike_times)
    rates = np.empty(len(starts))
    for i, (t0, t1) in enumerate(zip(starts, stops)):
        rates[i] = np.sum((spike_times >= t0) & (spike_times < t1)) / (t1 - t0)
    return rates


fr_matrix = np.column_stack([
    trial_interval_firing_rate(st, window_starts, window_stops) for st in units['spike_times']
])
fr_matrix_df = pd.DataFrame(fr_matrix, columns=units.index, index=reg_trials.index)
print(f"per-trial firing rates: {fr_matrix.shape[0]} trials x {fr_matrix.shape[1]} units")

context_labels = reg_trials['rewarded_modality'].values
block_labels = reg_trials['block_index'].values
print(f"contexts: {dict(pd.Series(context_labels).value_counts())}, "
      f"{reg_trials['block_index'].nunique()} blocks")

# Areas included (qualified cells)

Identical to the 3-regressor notebook: QC-pass single units, areas with at least
`MIN_UNITS` of them, each subsampled to exactly `MIN_UNITS` with `RNG_SEED = 0`. With
the same seed and the same `sua` table this yields the same units as that notebook.

In [ ]:
MIN_UNITS = 30
SUBSAMPLE = True
RNG_SEED = 0

sua = units[units['is_qc_pass'] & (units['decoder_label'] == 'sua')]
area_unit_counts = sua['structure'].value_counts()
areas = area_unit_counts[area_unit_counts >= MIN_UNITS].index.tolist()

rng = npr.default_rng(RNG_SEED)


def select_area_units(area):
    unit_ids = sua[sua['structure'] == area].index.values
    if SUBSAMPLE:
        unit_ids = rng.choice(unit_ids, size=MIN_UNITS, replace=False)
    return unit_ids


area_units = {area: select_area_units(area) for area in areas}
pairs = [(s, t) for s in areas for t in areas if s != t]

print(f"{len(areas)} areas with >= {MIN_UNITS} QC-pass SUA units"
      f"{f' (subsampled to {MIN_UNITS} each)' if SUBSAMPLE else ''}; {len(pairs)} ordered pairs")
for area in areas:
    print(f"  {area}: {len(area_units[area])} units (of {area_unit_counts[area]} available)")

# Residual variants

`base_activity[area]` is the z-scored quiescent firing rate (trials x units). Each
variant subtracts, per unit, the mean within the groups defined by its condition
columns (none = grand mean only, which is already zero after z-scoring). The table
below reports the fraction of each area's total variance removed by that
subtraction -- how much of the quiescent-period variance the condition means account
for in the first place.

Add or edit entries in `RESIDUAL_VARIANTS` to try other definitions (e.g.
`['block_index']` removes per-block means, i.e. context plus slow within-session
drift).

In [ ]:
RESIDUAL_VARIANTS = {
    'grand_mean': [],
    'context_mean': ['rewarded_modality'],
}
variants = list(RESIDUAL_VARIANTS)

base_activity = {area: StandardScaler().fit_transform(fr_matrix_df[area_units[area]].values) for area in areas}


def subtract_condition_mean(activity, condition_cols):
    """activity minus the per-unit mean within each group of trials defined by
    condition_cols (grand mean only if condition_cols is empty)."""
    activity = np.asarray(activity, dtype=float)
    if not condition_cols:
        return activity - activity.mean(axis=0)
    groups = reg_trials.groupby(condition_cols, observed=True).indices
    residual = np.empty_like(activity)
    for idx in groups.values():
        residual[idx] = activity[idx] - activity[idx].mean(axis=0)
    return residual


residual_activity = {
    variant: {area: subtract_condition_mean(base_activity[area], cols) for area in areas}
    for variant, cols in RESIDUAL_VARIANTS.items()
}

variance_removed = pd.DataFrame({
    variant: {
        area: 1 - residual_activity[variant][area].var(axis=0).sum() / base_activity[area].var(axis=0).sum()
        for area in areas
    }
    for variant in variants
}).reindex(areas)

for variant, cols in RESIDUAL_VARIANTS.items():
    n_groups = len(reg_trials.groupby(cols, observed=True)) if cols else 1
    print(f"{variant:>20}: condition = {cols or 'none'} ({n_groups} group(s))")
print("\nfraction of total variance removed by the condition-mean subtraction:")
variance_removed.round(4)

# 1. Contextual axis per area (LDA on the grand-mean activity)

One unit-normalized LDA axis per area, fit on `grand_mean` activity (context retained),
shared across all variants. Training accuracy and leave-one-block-out cross-validated
accuracy are reported as diagnostics. The last column of the table is the variance of
each variant's residual along that axis, as a fraction of the `grand_mean` variance
along it -- a direct check of how much "context signal" each subtraction removes from
the very direction being compared to the communication subspaces.

In [ ]:
def contextual_axis(activity, labels):
    lda = LinearDiscriminantAnalysis(solver='svd').fit(activity, labels)
    axis = lda.coef_.ravel()
    return axis / np.linalg.norm(axis), lda.score(activity, labels)


def contextual_axis_cv_accuracy(activity, labels, groups):
    correct = total = 0
    for train_idx, test_idx in LeaveOneGroupOut().split(activity, labels, groups):
        lda = LinearDiscriminantAnalysis(solver='svd').fit(activity[train_idx], labels[train_idx])
        correct += (lda.predict(activity[test_idx]) == labels[test_idx]).sum()
        total += len(test_idx)
    return correct / total


context_axes, context_accuracy, context_cv_accuracy = {}, {}, {}
for area in areas:
    context_axes[area], context_accuracy[area] = contextual_axis(base_activity[area], context_labels)
    context_cv_accuracy[area] = contextual_axis_cv_accuracy(base_activity[area], context_labels, block_labels)

axis_variance = pd.DataFrame({
    f'var. along axis ({variant}) / grand_mean': {
        area: (residual_activity[variant][area] @ context_axes[area]).var()
              / (residual_activity['grand_mean'][area] @ context_axes[area]).var()
        for area in areas
    }
    for variant in variants[1:]
})

context_summary = pd.concat([
    pd.DataFrame({'train_accuracy': context_accuracy, 'loo_block_cv_accuracy': context_cv_accuracy}),
    axis_variance,
], axis=1).reindex(areas)
context_summary.round(3)

# 2. Communication subspaces, every ordered pair, every variant

Same estimator as the 3-regressor notebook (ridge from source to target, alpha chosen
by efficient leave-one-out `RidgeCV`, SVD of the prediction, smallest number of
components reaching `VARIANCE_THRESHOLD` of predictable variance, orthonormalized in
source-unit space), with one deliberate difference: the input residuals are centered
but **not** re-standardized, so every subspace shares the contextual axis's coordinate
system (see intro). The LOO-CV R² is the predictive performance of the ridge fit.

In [ ]:
ALPHA_GRID = np.logspace(-1, 4, 25)
VARIANCE_THRESHOLD = 0.90


def communication_subspace_from_alpha(X_pop, Y_pop, alpha, threshold=VARIANCE_THRESHOLD):
    ridge = Ridge(alpha=alpha, fit_intercept=False).fit(X_pop, Y_pop)
    B_ridge = ridge.coef_.T
    Y_hat = X_pop @ B_ridge
    _, S, Vt = np.linalg.svd(Y_hat, full_matrices=False)
    component_variance_ratio = S ** 2 / np.sum(S ** 2)
    n_dims = int(np.searchsorted(np.cumsum(component_variance_ratio), threshold) + 1)
    Q_comm, _ = np.linalg.qr(B_ridge @ Vt[:n_dims].T)
    return Q_comm[:, :n_dims], n_dims, component_variance_ratio


def communication_subspace_from_activity(X_pop, Y_pop, alpha_grid=ALPHA_GRID, threshold=VARIANCE_THRESHOLD):
    """X_pop, Y_pop: already-centered residual activity (no re-scaling here)."""
    ridge_cv = RidgeCV(alphas=alpha_grid, alpha_per_target=False, scoring='r2', fit_intercept=False)
    ridge_cv.fit(X_pop, Y_pop)
    Q_comm, n_dims, cvr = communication_subspace_from_alpha(X_pop, Y_pop, ridge_cv.alpha_, threshold)
    return Q_comm, n_dims, ridge_cv.best_score_, ridge_cv.alpha_, cvr


def loo_r2_at_alpha(X_pop, Y_pop, alpha):
    return RidgeCV(alphas=[alpha], alpha_per_target=False, scoring='r2', fit_intercept=False).fit(X_pop, Y_pop).best_score_


def empty_matrix():
    return pd.DataFrame(index=areas, columns=areas, dtype=float)


comm_subspaces = {v: {} for v in variants}
comm_alpha = {v: {} for v in variants}
n_dims_matrix = {v: empty_matrix() for v in variants}
r2_matrix = {v: empty_matrix() for v in variants}

for variant in variants:
    act = residual_activity[variant]
    for s, t in tqdm(pairs, desc=f'communication subspaces [{variant}]'):
        Q, n_dims, r2, alpha, _ = communication_subspace_from_activity(act[s], act[t])
        comm_subspaces[variant][(s, t)] = Q
        comm_alpha[variant][(s, t)] = alpha
        n_dims_matrix[variant].loc[s, t] = n_dims
        r2_matrix[variant].loc[s, t] = r2

pd.concat({v: r2_matrix[v].round(3) for v in variants}, axis=1)

In [ ]:
PANEL_LETTER_PT = 14


def label_panels(fig, axes):
    """Bold panel letters A, B, ... in reading order (figures with one panel get none). Each letter is left-aligned with
    its panel's y-axis label, letters are snapped to shared columns, and each letter's cap top is level with the cap top
    of its panel title (shared across a row). Call after the layout is final (after tight_layout)."""
    axes = list(np.ravel(axes))
    if len(axes) < 2:
        return
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()

    def cap_height(fp):
        _, h, d = r.get_text_width_height_descent('H', fp, ismath=False)
        return h - d

    letter_fp = plt.matplotlib.font_manager.FontProperties(size=PANEL_LETTER_PT, weight='bold')
    xs, baselines = [], []
    for ax in axes:
        yl = ax.yaxis.label
        xs.append((yl.get_window_extent(r) if yl.get_text() else ax.get_tightbbox(r)).x0)
        title = ax.title
        if title.get_text():
            first = title.get_text().split('\n')[0]
            _, h, d = r.get_text_width_height_descent(first, title.get_fontproperties(), ismath=False)
            cap_top = title.get_window_extent(r).y1 - (h - d) + cap_height(title.get_fontproperties())
        else:
            cap_top = ax.get_window_extent(r).y1 + 4 * fig.dpi / 72 + cap_height(letter_fp)
        baselines.append(cap_top - cap_height(letter_fp))
    xs, baselines = np.array(xs), np.array(baselines)

    def snap(vals, tol, agg):
        out, order = vals.copy(), np.argsort(vals)
        groups = []
        for i in order:
            if groups and vals[i] - vals[groups[-1][-1]] <= tol:
                groups[-1].append(i)
            else:
                groups.append([i])
        for g in groups:
            out[g] = agg(vals[g])
        return out

    xs = snap(xs, 0.05 * fig.bbox.width, np.min)
    baselines = snap(baselines, 0.03 * fig.bbox.height, np.max)
    for i, (x, y) in enumerate(zip(xs, baselines)):
        fx, fy = inv.transform((x, y))
        fig.text(fx, fy, chr(ord('A') + i), fontproperties=letter_fp, ha='left', va='baseline')


def tidy_text(s):
    """Plot text: variant identifiers as words (grand_mean -> grand mean), first letter of every line capitalized."""
    for v in RESIDUAL_VARIANTS:
        s = s.replace(v, v.replace('_', ' '))
    return '\n'.join(line[:1].upper() + line[1:] for line in s.split('\n'))


def finish_figure(fig, axes):
    """Capitalize titles / axis / colorbar / legend labels, resolve the layout, then add aligned panel letters."""
    if fig._suptitle is not None:
        fig._suptitle.set_text(tidy_text(fig._suptitle.get_text()))
    for ax in fig.axes:
        for text in (ax.title, ax.xaxis.label, ax.yaxis.label):  # set_text keeps each label's font size
            text.set_text(tidy_text(text.get_text()))
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title(tidy_text(legend.get_title().get_text()))
    plt.tight_layout()
    label_panels(fig, axes)


def heatmap_grid(matrices, row_label, fmt, cmap, vmin=None, vmax=None, cbar_label='', title='', mask_na=True):
    """One heatmap per variant, in a row, on a shared color scale."""
    n = len(matrices)
    fig, axes = plt.subplots(1, n, figsize=(0.8 + 0.65 * len(areas) * n, 0.65 * len(areas) + 0.6), squeeze=False)
    diag = np.eye(len(areas), dtype=bool)
    all_vals = np.concatenate([m.values[~diag] for m in matrices.values()])
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin = np.nanmin(all_vals) if vmin is None else vmin
    vmax = np.nanmax(all_vals) if vmax is None else vmax
    for ax, (variant, m) in zip(axes[0], matrices.items()):
        mask = diag | (m.isna().values if mask_na else False)
        sns.heatmap(m.astype(float), mask=mask, annot=True, fmt=fmt, cmap=cmap, vmin=vmin, vmax=vmax,
                    square=True, cbar_kws={'label': cbar_label}, ax=ax)
        ax.set_title(variant, fontsize=10)
        ax.set_xlabel('target area')
        ax.set_ylabel(row_label)
        ax.tick_params(labelsize=8)
    fig.suptitle(f'{title}\nsession {session_id} | quiescent window', fontsize=11)
    finish_figure(fig, axes[0])
    plt.show()


heatmap_grid(n_dims_matrix, 'source area', '.0f', 'mako', vmin=0,
             cbar_label='number of dimensions', title='Communication subspace dimensionality (90% of predictable variance)')
heatmap_grid(r2_matrix, 'source area', '.3f', 'viridis', vmin=0,
             cbar_label='predictive R² (LOO-CV)', title='Communication subspace predictive R²')

# 3. Trial-shuffle test: is each subspace's R² above chance? (per variant)

For every pair and variant, shuffle the target trials relative to the source trials and
recompute the LOO-CV R² at the pair's selected alpha, `N_R2_PERMUTATIONS` times.
Pairs whose FDR-BH-corrected p-value is below `FDR_ALPHA` form `significant_pairs[variant]`;
only those are carried into the cosine-similarity comparison for that variant.

In [ ]:
N_R2_PERMUTATIONS = 1000
FDR_ALPHA = 0.05


def fdr_bh(pvals, alpha=FDR_ALPHA):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    corrected = np.clip(np.minimum.accumulate((pvals[order] * n / np.arange(1, n + 1))[::-1])[::-1], 0, 1)
    out = np.empty(n)
    out[order] = corrected
    return out <= alpha, out


def pvals_to_matrices(pair_list, raw):
    """(raw, FDR-BH) p-value matrices from a list of raw p-values aligned to pair_list."""
    _, fdr = fdr_bh(raw)
    raw_m, fdr_m = empty_matrix(), empty_matrix()
    for (s, t), p, q in zip(pair_list, raw, fdr):
        raw_m.loc[s, t], fdr_m.loc[s, t] = p, q
    return raw_m, fdr_m


r2_p_raw, r2_p_fdr, significant_pairs = {}, {}, {}
for variant in variants:
    act = residual_activity[variant]
    perm_rng = npr.default_rng(RNG_SEED + 3)  # same shuffles for every variant
    raw = []
    for s, t in tqdm(pairs, desc=f'R2 trial-shuffle nulls [{variant}]'):
        X_pop, Y_pop = act[s], act[t]
        null = np.array([loo_r2_at_alpha(X_pop, Y_pop[perm_rng.permutation(len(Y_pop))], comm_alpha[variant][(s, t)])
                         for _ in range(N_R2_PERMUTATIONS)])
        raw.append((np.sum(null >= r2_matrix[variant].loc[s, t]) + 1) / (N_R2_PERMUTATIONS + 1))
    r2_p_raw[variant], r2_p_fdr[variant] = pvals_to_matrices(pairs, raw)
    significant_pairs[variant] = [(s, t) for s, t in pairs if r2_p_fdr[variant].loc[s, t] <= FDR_ALPHA]
    print(f"{variant:>20}: {len(significant_pairs[variant])}/{len(pairs)} ordered pairs significant at FDR q < {FDR_ALPHA}")

heatmap_grid(r2_p_fdr, 'source area', '.3f', 'viridis', vmin=0, vmax=1,
             cbar_label='FDR-BH p-value', title=f'Communication R² significance ({N_R2_PERMUTATIONS} trial shuffles)')

# 4. Cosine similarity: contextual axis vs. communication subspace, per variant

Norm of the source area's (shared) contextual axis projected onto that pair's
communication subspace. 1 = the axis lies within the subspace, 0 = orthogonal to it.
Blank cells did not pass the R² test for that variant.

In [ ]:
cosine_matrix = {}
for variant in variants:
    m = empty_matrix()
    for s, t in significant_pairs[variant]:
        m.loc[s, t] = np.linalg.norm(comm_subspaces[variant][(s, t)].T @ context_axes[s])
    cosine_matrix[variant] = m

heatmap_grid(cosine_matrix, 'source area (contextual axis)', '.2f', 'viridis', vmin=0, vmax=1,
             cbar_label='cosine similarity', title='Contextual axis vs. communication subspace')
pd.concat({v: cosine_matrix[v].round(3) for v in variants}, axis=1)

# 5. How much does the alignment change when the condition mean is subtracted?

Three views of the same paired comparison, each restricted to pairs significant in
*both* variants being compared:

- **Difference heatmaps**: cosine(variant) minus cosine(`grand_mean`).
- **Paired scatter**: every pair as a point, `grand_mean` on x and the subtracted
  variant on y, colored by source area. Points below the diagonal are pairs whose
  contextual axis became *less* aligned with the communication subspace once the
  condition mean was removed.
- **Summary table**: mean +- SEM of the paired difference, the fraction of pairs that
  decreased, and a two-sided Wilcoxon signed-rank test across pairs. (Pairs are not
  independent -- they share areas -- so treat the p-value as descriptive.)

Interpretation. If subtracting the context mean shrinks the cosine similarity, the
alignment seen with `grand_mean` was carried in part by the across-context *mean*
difference: the context signal is literally part of what the source predicts about the
target. If the similarity survives, the alignment lives in trial-to-trial covariability
within a context, which is the sense in which "communication subspace" is usually
meant. The subspace-overlap diagnostic in section 6 says whether the subspace itself
moved or only the axis's projection onto it did.

In [ ]:
ref = variants[0]
comparisons = [(ref, v) for v in variants[1:]]

diff_matrix = {}
rows = []
for a, b in comparisons:
    both = [p for p in significant_pairs[a] if p in significant_pairs[b]]
    d = empty_matrix()
    for s, t in both:
        d.loc[s, t] = cosine_matrix[b].loc[s, t] - cosine_matrix[a].loc[s, t]
    diff_matrix[f'{b} - {a}'] = d
    x = np.array([cosine_matrix[a].loc[s, t] for s, t in both])
    y = np.array([cosine_matrix[b].loc[s, t] for s, t in both])
    if len(both) >= 2 and np.any(y != x):
        w = stats.wilcoxon(y, x, alternative='two-sided')
        w_stat, w_p = w.statistic, w.pvalue
    else:
        w_stat, w_p = np.nan, np.nan
    rows.append({
        'comparison': f'{b} vs {a}', 'n_pairs': len(both),
        f'mean cosine ({a})': x.mean() if len(x) else np.nan,
        f'mean cosine (variant)': y.mean() if len(y) else np.nan,
        'mean diff': (y - x).mean() if len(x) else np.nan,
        'sem diff': (y - x).std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan,
        'fraction decreased': np.mean(y < x) if len(x) else np.nan,
        'wilcoxon W': w_stat, 'wilcoxon p': w_p,
    })
comparison_summary = pd.DataFrame(rows).set_index('comparison')

lim = np.nanmax(np.concatenate([diff_matrix[k].abs().values.ravel() for k in diff_matrix] + [np.array([0.05])]))
heatmap_grid(diff_matrix, 'source area (contextual axis)', '.2f', 'RdBu_r', vmin=-lim, vmax=lim,
             cbar_label='Δ cosine similarity', title='Change in cosine similarity vs. grand_mean')

fig, axes = plt.subplots(1, len(comparisons), figsize=(4.2 * len(comparisons), 4), squeeze=False)
palette = dict(zip(areas, sns.color_palette('tab10', len(areas))))
for ax, (a, b) in zip(axes[0], comparisons):
    both = [p for p in significant_pairs[a] if p in significant_pairs[b]]
    for s, t in both:
        ax.scatter(cosine_matrix[a].loc[s, t], cosine_matrix[b].loc[s, t], color=palette[s], s=36,
                   edgecolor='k', linewidth=0.4, label=s)
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xlabel(f'cosine similarity ({a})')
    ax.set_ylabel(f'cosine similarity ({b})')
    ax.set_title(f'{b} vs {a} (n = {len(both)} pairs)', fontsize=10)
handles, labels = axes[0][0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[0][-1].legend(by_label.values(), by_label.keys(), title='source area', fontsize=8, title_fontsize=8,
                   bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle(f'Paired cosine similarity per area pair\nsession {session_id} | quiescent window')
finish_figure(fig, axes[0])
plt.show()

comparison_summary.round(4).assign(**{'wilcoxon p': comparison_summary['wilcoxon p'].map('{:.2e}'.format)})

# 6. Where did the alignment go? Subspace overlap and the raw mean-difference direction

Two diagnostics that separate "the subspace changed" from "the axis is no longer in
it":

- **Subspace overlap** (first figure): mean squared cosine of the principal angles
  between the `grand_mean` subspace and each variant's subspace,
  `||Q_b^T Q_a||_F^2 / min(n_dims_a, n_dims_b)`. 1 = the smaller subspace is nested in
  the larger one; 0 = orthogonal. High overlap means the set of predictive source
  directions is largely the same in both variants.
- **Raw context mean-difference direction** `m` (second figure and table): per area,
  the unit vector from the auditory-context mean to the visual-context mean in the base
  (z-scored) unit space. Unlike the LDA axis, `m` is *not* whitened by the
  within-context covariance. The heatmaps show `||Q^T m||` for every pair and variant;
  the table shows how much each area's LDA axis agrees with `m`, and how much variance
  each variant's residual still carries along `m`.

Why this matters: the LDA axis is `Σ_w^{-1} m` (within-context covariance inverse times
the mean difference). If `m` remains inside the communication subspace after the
context mean is subtracted, then the across-context shift happens along a direction
the two areas *also* share trial to trial within a context -- and a drop in the LDA
axis's cosine means it is the whitening, not the mean shift itself, that takes the
axis out of the subspace. If `m` leaves the subspace too, the alignment was carried by
the across-context mean difference alone. Either way, the result depends on how the
"task axis" is defined, which is worth keeping in mind when comparing variants.

In [ ]:
subspace_overlap = {}
for a, b in comparisons:
    ov = empty_matrix()
    for s, t in pairs:
        Qa, Qb = comm_subspaces[a][(s, t)], comm_subspaces[b][(s, t)]
        ov.loc[s, t] = np.linalg.norm(Qb.T @ Qa, 'fro') ** 2 / min(Qa.shape[1], Qb.shape[1])
    subspace_overlap[f'{b} vs {a}'] = ov

heatmap_grid(subspace_overlap, 'source area', '.2f', 'viridis', vmin=0, vmax=1, mask_na=False,
             cbar_label='mean squared principal-angle cosine',
             title='Overlap between communication subspaces across residual variants (all pairs)')

# raw context mean-difference direction per area, in base (z-scored) unit space
mean_diff_dir = {}
for area in areas:
    d = base_activity[area][context_labels == 'vis'].mean(axis=0) - base_activity[area][context_labels == 'aud'].mean(axis=0)
    mean_diff_dir[area] = d / np.linalg.norm(d)

mean_diff_cosine = {}
for variant in variants:
    m = empty_matrix()
    for s, t in pairs:
        m.loc[s, t] = np.linalg.norm(comm_subspaces[variant][(s, t)].T @ mean_diff_dir[s])
    mean_diff_cosine[variant] = m

heatmap_grid(mean_diff_cosine, 'source area (mean-difference dir.)', '.2f', 'viridis', vmin=0, vmax=1, mask_na=False,
             cbar_label='||Qᵀ m||',
             title='Raw context mean-difference direction vs. communication subspace (all pairs)')

mean_diff_table = pd.DataFrame({
    'cos(LDA axis, m)': {a: abs(context_axes[a] @ mean_diff_dir[a]) for a in areas},
    **{f'var along m ({v}) / grand_mean': {
        a: (residual_activity[v][a] @ mean_diff_dir[a]).var() / (residual_activity['grand_mean'][a] @ mean_diff_dir[a]).var()
        for a in areas} for v in variants[1:]},
    **{f'mean ||Qᵀ m|| over targets ({v})': mean_diff_cosine[v].mean(axis=1) for v in variants},
    **{f'mean ||Qᵀ LDA axis|| over targets ({v})': cosine_matrix[v].mean(axis=1) for v in variants},
}).reindex(areas)
mean_diff_table.round(3)

# 7. Permutation nulls for the cosine similarity, per variant

The same three nulls as the 3-regressor notebook, each applied to every variant's
significant pairs, with FDR-BH correction across those pairs:

- **Label shuffle**: refit the LDA axis on `grand_mean` activity with shuffled context
  labels (`N_PERMUTATIONS` times; the same null axes serve every variant, since the
  axis is shared), project onto the real subspace.
- **Trial shuffle**: hold the axis fixed, shuffle target trials relative to source
  trials, refit the subspace at the real alpha, project the real axis onto it.
- **Random axis**: replace the contextual axis by a uniformly random unit vector in the
  source area's unit space, project onto the real subspace. This is the pure
  dimensionality baseline: a random direction's projection norm grows with `n_dims`.

The grid shows FDR-corrected p-values (rows = null, columns = variant); the table
counts significant pairs per null and variant.

In [ ]:
N_PERMUTATIONS = 1000
N_TRIAL_SHUFFLE_PERMUTATIONS = 1000
N_RANDOM_AXIS_PERMUTATIONS = 1000


def permuted_contextual_axes(area, n, rng):
    activity = base_activity[area]
    lda = LinearDiscriminantAnalysis(solver='svd')
    out = np.empty((n, activity.shape[1]))
    for i in range(n):
        lda.fit(activity, rng.permutation(context_labels))
        out[i] = lda.coef_.ravel() / np.linalg.norm(lda.coef_)
    return out


def random_unit_vectors(d, n, rng):
    v = rng.standard_normal((n, d))
    return v / np.linalg.norm(v, axis=1, keepdims=True)


sources = sorted({s for v in variants for s, t in significant_pairs[v]})
label_rng = npr.default_rng(RNG_SEED)
random_axis_rng = npr.default_rng(RNG_SEED + 4)
null_axes = {a: permuted_contextual_axes(a, N_PERMUTATIONS, label_rng) for a in tqdm(sources, desc='label-shuffle axes')}
random_axes = {a: random_unit_vectors(base_activity[a].shape[1], N_RANDOM_AXIS_PERMUTATIONS, random_axis_rng) for a in sources}

null_names = ['label shuffle', 'trial shuffle', 'random axis']
p_fdr = {name: {} for name in null_names}
null_count_rows = []
for variant in variants:
    sig = significant_pairs[variant]
    raw = {name: [] for name in null_names}
    trial_rng = npr.default_rng(RNG_SEED + 2)
    for s, t in tqdm(sig, desc=f'cosine nulls [{variant}]'):
        Q = comm_subspaces[variant][(s, t)]
        observed = cosine_matrix[variant].loc[s, t]
        axis = context_axes[s]

        null_label = np.linalg.norm(null_axes[s] @ Q, axis=1)
        raw['label shuffle'].append((np.sum(null_label >= observed) + 1) / (N_PERMUTATIONS + 1))

        X_pop, Y_pop = residual_activity[variant][s], residual_activity[variant][t]
        alpha = comm_alpha[variant][(s, t)]
        null_trial = np.empty(N_TRIAL_SHUFFLE_PERMUTATIONS)
        for i in range(N_TRIAL_SHUFFLE_PERMUTATIONS):
            Q_null, _, _ = communication_subspace_from_alpha(X_pop, Y_pop[trial_rng.permutation(len(Y_pop))], alpha)
            null_trial[i] = np.linalg.norm(Q_null.T @ axis)
        raw['trial shuffle'].append((np.sum(null_trial >= observed) + 1) / (N_TRIAL_SHUFFLE_PERMUTATIONS + 1))

        null_random = np.linalg.norm(random_axes[s] @ Q, axis=1)
        raw['random axis'].append((np.sum(null_random >= observed) + 1) / (N_RANDOM_AXIS_PERMUTATIONS + 1))

    row = {'variant': variant, 'n_pairs (R² significant)': len(sig)}
    for name in null_names:
        _, p_fdr[name][variant] = pvals_to_matrices(sig, raw[name]) if sig else (empty_matrix(), empty_matrix())
        row[f'{name}: sig. at FDR q<{FDR_ALPHA}'] = int((p_fdr[name][variant].values <= FDR_ALPHA).sum())
    null_count_rows.append(row)

null_counts = pd.DataFrame(null_count_rows).set_index('variant')

fig, axes = plt.subplots(len(null_names), len(variants),
                         figsize=(0.8 + 0.65 * len(areas) * len(variants), (0.65 * len(areas) + 0.5) * len(null_names)),
                         squeeze=False)
diag = np.eye(len(areas), dtype=bool)
for i, name in enumerate(null_names):
    for j, variant in enumerate(variants):
        ax = axes[i, j]
        m = p_fdr[name][variant]
        sns.heatmap(m.astype(float), mask=diag | m.isna().values, annot=True, fmt='.3f', cmap='viridis',
                    vmin=0, vmax=1, square=True, cbar_kws={'label': 'FDR-BH p'}, ax=ax)
        ax.set_title(f'{name} | {variant}', fontsize=10)
        ax.set_xlabel('target area')
        ax.set_ylabel('source area (contextual axis)')
        ax.tick_params(labelsize=8)
fig.suptitle(f'Cosine-similarity permutation tests, FDR-corrected\nsession {session_id} | quiescent window', fontsize=11)
finish_figure(fig, axes)
plt.show()

null_counts

# 8. One-table summary

Per variant: how many pairs had a significant communication subspace, the mean cosine
similarity of the LDA contextual axis over those pairs, the same for the raw
mean-difference direction `m` (section 6, diagnostic only -- no nulls were run for
it), the mean random-axis baseline (what an arbitrary direction would score given each
subspace's dimensionality), and how many pairs beat each null.

In [ ]:
summary_rows = []
for variant in variants:
    sig = significant_pairs[variant]
    cos = np.array([cosine_matrix[variant].loc[s, t] for s, t in sig])
    rand_baseline = np.array([np.linalg.norm(random_axes[s] @ comm_subspaces[variant][(s, t)], axis=1).mean() for s, t in sig])
    nd = np.array([n_dims_matrix[variant].loc[s, t] for s, t in sig])
    summary_rows.append({
        'variant': variant,
        'condition cols': ', '.join(RESIDUAL_VARIANTS[variant]) or 'none',
        'n sig. pairs': len(sig),
        'mean n_dims': nd.mean() if len(nd) else np.nan,
        'mean R² (LOO)': np.mean([r2_matrix[variant].loc[s, t] for s, t in sig]) if sig else np.nan,
        'mean cosine': cos.mean() if len(cos) else np.nan,
        'mean cosine (raw mean-diff dir.)': np.mean([mean_diff_cosine[variant].loc[s, t] for s, t in sig]) if sig else np.nan,
        'mean random-axis baseline': rand_baseline.mean() if len(rand_baseline) else np.nan,
        **{f'beat {name} (FDR)': int((p_fdr[name][variant].values <= FDR_ALPHA).sum()) for name in null_names},
    })
summary = pd.DataFrame(summary_rows).set_index('variant')
summary.round(3)